# SSN GFM load-step benchmark results

This notebook plots the 50 µs simulation output, the network topology, and
where the measured simulation time is spent. Run `EMT_SSN_GFM_Test` from the
DPsim repository root first in both CSV and real-time logger modes. The notebook reads the buffered real-time logger results from
`logs/EMT_SSN_GFM_LoadStep_EMT/` and the conventional CSV logger comparison
from `logs/EMT_SSN_GFM_LoadStep_EMT_CSVLogger/`. When using the supplied ZIP,
extract it and run this notebook from the extracted folder; its bundled logs are
already in those relative locations.

The configured horizon is 3 s. Both loads use EMT SSN fixed-impedance models.
The extra 3 kW resistive load is switched on at 1 s and off at 2 s.

In [ ]:
from pathlib import Path
import re

import matplotlib.pyplot as plt
from matplotlib.patches import Circle, Rectangle
import numpy as np
import pandas as pd

# Find the benchmark package root when launched from the package or a child folder.
repo_root = next(
    (
        p
        for p in [Path.cwd(), *Path.cwd().parents]
        if (p / "logs/EMT_SSN_GFM_LoadStep_EMT/EMT_SSN_GFM_LoadStep_EMT.csv").exists()
    ),
    None,
)
if repo_root is None:
    raise FileNotFoundError(
        "Could not find bundled logs. Extract the benchmark ZIP and run this notebook from its folder, or run the benchmark first from the DPsim repository root."
    )
log_dir = repo_root / "logs/EMT_SSN_GFM_LoadStep_EMT"
results = pd.read_csv(log_dir / "EMT_SSN_GFM_LoadStep_EMT.csv", skipinitialspace=True)
results.columns = results.columns.str.strip()
task_times = pd.read_csv(log_dir / "per_step_task_timing.csv", skipinitialspace=True)
task_times.columns = task_times.columns.str.strip()
component_times = pd.read_csv(
    log_dir / "per_step_component_timing.csv", skipinitialspace=True
)
performance_data = pd.read_csv(log_dir / "performance_summary.csv")
performance = dict(
    zip(performance_data.metric, pd.to_numeric(performance_data.value, errors="coerce"))
)
csv_dir = repo_root / "logs/EMT_SSN_GFM_LoadStep_EMT_CSVLogger"
csv_performance_data = pd.read_csv(csv_dir / "performance_summary.csv")
csv_performance = dict(
    zip(
        csv_performance_data.metric,
        pd.to_numeric(csv_performance_data.value, errors="coerce"),
    )
)
csv_component_times = pd.read_csv(csv_dir / "per_step_component_timing.csv")
csv_task_times = pd.read_csv(csv_dir / "per_step_task_timing.csv")
solver_log = (log_dir / "EMT_SSN_GFM_LoadStep_EMT_Solver.log").read_text()
csv_simulation_log = (csv_dir / "EMT_SSN_GFM_LoadStep_EMT_CSVLogger.log").read_text()
realtime_simulation_log = (log_dir / "EMT_SSN_GFM_LoadStep_EMT.log").read_text()


def calculation_time(log_text):
    match = re.search(r"Simulation calculation time: ([0-9.eE+-]+)", log_text)
    if match is None:
        raise ValueError("Simulation log does not contain calculation time")
    return float(match.group(1))


csv_calculation_s = calculation_time(csv_simulation_log)
realtime_calculation_s = calculation_time(realtime_simulation_log)


def solver_metric(pattern):
    match = re.search(pattern, solver_log)
    if match is None:
        raise ValueError(f"Could not find solver timing matching {pattern!r}")
    return float(match.group(1))


refactorization_s = solver_metric(r"Cumulative refactorization times: ([0-9.eE+-]+)")
linear_solve_s = solver_metric(r"Cumulative solve times: ([0-9.eE+-]+)")
executed_steps = int(performance.get("number_of_executed_steps", len(task_times)))
assert len(results) == 60_001
assert len(task_times) == executed_steps
assert len(component_times) == executed_steps
assert len(csv_component_times) == executed_steps
assert len(csv_task_times) == executed_steps
assert results.columns[0] == "time"
gfm_columns = [name for name in results.columns if name.startswith("GFM_")]
assert list(results.columns[1 : len(gfm_columns) + 1]) == gfm_columns
print(
    f"Samples: {len(results):,}; executed steps: {executed_steps:,}; "
    f"time step: {performance['time_step'] * 1e6:.0f} µs; "
    f"simulation wall time: {performance['simulation_wall_clock']:.3f} s"
)

## Network topology

The SSN GFM converter feeds the main bus through an SSN PiLine. The SSN base
load is connected at that bus. An SSN load switch connects the additional SSN
step load from 1 s until 2 s.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.3))
ax.set_xlim(0, 1)
ax.set_ylim(0.05, 0.98)
ax.axis("off")


def block(x, y, w, h, title, detail):
    rect = Rectangle(
        (x, y), w, h, facecolor="#edf3f8", edgecolor="#29465b", linewidth=1.5
    )
    ax.add_patch(rect)
    ax.text(x + w / 2, y + h * 0.62, title, ha="center", va="center", weight="bold")
    ax.text(x + w / 2, y + h * 0.30, detail, ha="center", va="center", fontsize=9)


# Main feeder: SSN GFM -> SSN PiLine -> load bus.
block(0.03, 0.53, 0.19, 0.26, "SSN GFM", "50 Hz voltage source")
ax.plot([0.22, 0.29], [0.66, 0.66], color="#29465b", lw=2)
block(0.29, 0.55, 0.19, 0.22, "SSN PiLine", "R = 0.20 Ω · L = 0.1 mH\nC = 0.1 µF/phase")
ax.plot([0.48, 0.62], [0.66, 0.66], color="#29465b", lw=2)
ax.plot([0.62, 0.62], [0.58, 0.75], color="#29465b", lw=3)
ax.add_patch(Circle((0.62, 0.66), 0.012, color="#29465b"))
ax.text(0.62, 0.81, "SSN load bus", ha="center", va="bottom", weight="bold")

# Base load shunt.
ax.plot([0.62, 0.62], [0.66, 0.39], color="#29465b", lw=2)
block(0.52, 0.20, 0.20, 0.19, "SSN base load", "12 kW · R ≈ 12.1 Ω/phase")
ax.plot([0.62, 0.62], [0.20, 0.14], color="#29465b", lw=2)
for y, width in [(0.14, 0.10), (0.115, 0.065), (0.09, 0.03)]:
    ax.plot([0.62 - width / 2, 0.62 + width / 2], [y, y], color="#29465b", lw=1.5)

# Switched step-load branch.
ax.plot([0.62, 0.76], [0.66, 0.66], color="#29465b", lw=2)
ax.add_patch(Circle((0.76, 0.66), 0.009, color="#29465b"))
ax.plot([0.76, 0.81], [0.66, 0.71], color="#29465b", lw=2)
ax.add_patch(Circle((0.85, 0.66), 0.009, color="#29465b"))
ax.plot([0.85, 0.90], [0.66, 0.66], color="#29465b", lw=2)
block(0.90, 0.56, 0.095, 0.20, "SSN step load", "3 kW")
ax.text(0.81, 0.78, "SSN load switch", ha="center", va="bottom", fontsize=9)
ax.text(0.81, 0.48, "closed: 1–2 s", ha="center", va="top", fontsize=9)
fig.tight_layout()
plt.show()

In [ ]:
# Controller response and phase-A waveforms, with vertical load-step markers.
fig, axes = plt.subplots(2, 2, figsize=(14, 9), sharex=True)

ax = axes[0, 0]
ax.plot(results.time, results.GFM_ActivePower_W / 1000, label="Active power")
ax.plot(results.time, results.GFM_ReactivePower_var / 1000, label="Reactive power")
ax.set_ylabel("Power (kW / kvar)")
ax.set_title("SSN GFM power response")
ax.legend(loc="best")
ax.grid(True, alpha=0.3)

ax = axes[0, 1]
ax.plot(
    results.time,
    results.GFM_state_omega_rad_per_s / (2 * np.pi),
    label="SSN GFM frequency",
)
ax.axhline(50, color="black", linestyle="--", linewidth=1, label="50 Hz reference")
ax.set_ylabel("Frequency (Hz)")
ax.set_title("SSN GFM frequency state")
ax.legend(loc="best")
ax.grid(True, alpha=0.3)

ax = axes[1, 0]
ax.plot(results.time, results.GFM_BusVoltage_V_0, label="SSN GFM bus phase A")
ax.plot(results.time, results.BaseLoad_Voltage_V_0, label="SSN base-load bus phase A")
ax.plot(results.time, results.StepLoad_Voltage_V_0, label="SSN step-load bus phase A")
ax.set_ylabel("Instantaneous voltage (V)")
ax.set_title("SSN component phase-A voltages")
ax.set_xlabel("Time (s)")
ax.legend(loc="best")
ax.grid(True, alpha=0.3)

ax = axes[1, 1]
ax.plot(
    results.time, results.GFM_InterfaceCurrent_A_0, label="SSN GFM interface current"
)
ax.plot(results.time, results.StepLoad_Current_A_0, label="SSN step-load current")
ax.set_ylabel("Instantaneous current (A)")
ax.set_title("SSN component phase-A currents")
ax.set_xlabel("Time (s)")
ax.legend(loc="best")
ax.grid(True, alpha=0.3)

for ax in axes.flat:
    ax.axvline(1.0, color="tab:red", linestyle=":", linewidth=1.2)
    ax.axvline(2.0, color="tab:purple", linestyle=":", linewidth=1.2)
    ax.set_xlim(0, 3)
fig.suptitle("SSN GFM load step: 3 kW on at 1 s, off at 2 s", fontsize=15)
fig.tight_layout()
plt.show()

## Where the simulation time goes

The first bar chart compares SSN GFM component task time with the combined SSN base
and switched-load task time; the second breaks the load work down by load.
The per-step component chart shows pre-step and post-step work for the SSN GFM,
PiLine, and both SSN loads, plus the solver/integration step. A separate chart
compares the CSV and buffered real-time logger: time inside the simulation
loop, final file flush, and total run wall time. The matrix refactorization and linear-solve timers are measured
inside DPsim's solver task. Remaining solver-task time is shown separately so
those internal timings are not double-counted.

In [ ]:
gfm_component_s = component_times.GFM_total_s.sum()
pi_line_component_s = component_times.PiLine_total_s.sum()
base_load_component_s = component_times.BaseLoad_total_s.sum()
step_load_component_s = component_times.StepLoad_total_s.sum()
loads_component_s = component_times.Loads_total_s.sum()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
component_values = pd.Series(
    {
        "SSN GFM": gfm_component_s,
        "SSN PiLine": pi_line_component_s,
        "Both SSN loads": loads_component_s,
    }
).sort_values()
bars = axes[0].barh(
    component_values.index,
    component_values.values,
    color=["#fc8d62", "#66c2a5", "#7b6fd0"],
)
axes[0].set_title("SSN GFM vs. SSN load component time")
axes[0].set_xlabel("Cumulative component wall time (s)")
axes[0].grid(axis="x", alpha=0.3)
for bar, val in zip(bars, component_values.values):
    axes[0].text(
        val + max(component_values.values) * 0.02,
        bar.get_y() + bar.get_height() / 2,
        f"{val:.4f} s",
        va="center",
    )
axes[0].set_xlim(0, max(component_values.values) * 1.2)

load_values = pd.Series(
    {"SSN base load": base_load_component_s, "SSN step load": step_load_component_s}
).sort_values()
bars = axes[1].barh(load_values.index, load_values.values, color=["#8da0cb", "#e78ac3"])
axes[1].set_title("SSN load component time")
axes[1].set_xlabel("Cumulative component wall time (s)")
axes[1].grid(axis="x", alpha=0.3)
for bar, val in zip(bars, load_values.values):
    axes[1].text(
        val + max(load_values.values) * 0.02,
        bar.get_y() + bar.get_height() / 2,
        f"{val:.4f} s",
        va="center",
    )
axes[1].set_xlim(0, max(load_values.values) * 1.2)
fig.suptitle(f"Per-component task timings over {executed_steps:,} steps")
fig.tight_layout()
plt.show()

# Pre-step and post-step time by individual component.
component_names = ["GFM", "PiLine", "BaseLoad", "StepLoad"]
pre_totals = [
    component_times[f"{name}_pre_step_s"].sum() for name in component_names
] + [0.0]
post_totals = [
    component_times[f"{name}_post_step_s"].sum() for name in component_names
] + [0.0]
integration_totals = [0.0] * len(component_names) + [
    component_times.Solver_task_s.sum()
]
y = np.arange(len(component_names) + 1)
fig, ax = plt.subplots(figsize=(9, 5.5))
ax.barh(y, pre_totals, label="Pre-step", color="#4c78a8")
ax.barh(y, post_totals, left=pre_totals, label="Post-step", color="#f58518")
ax.barh(
    y,
    integration_totals,
    left=np.array(pre_totals) + np.array(post_totals),
    label="Solver / integration",
    color="#54a24b",
)
ax.set_yticks(
    y,
    [
        "SSN GFM",
        "SSN PiLine",
        "SSN base load",
        "SSN step load",
        "Solver / integration step",
    ],
)
ax.set_xlabel("Cumulative wall time (s)")
ax.set_title("Pre-step, post-step, and integration time")
ax.legend()
ax.grid(axis="x", alpha=0.3)
for row, total in enumerate(
    np.array(pre_totals) + np.array(post_totals) + np.array(integration_totals)
):
    ax.text(
        total
        + max(
            total
            for total in np.array(pre_totals)
            + np.array(post_totals)
            + np.array(integration_totals)
        )
        * 0.015,
        row,
        f"{total:.4f} s",
        va="center",
    )
fig.tight_layout()
plt.show()

# Non-overlapping solver-stage and component-stage totals.
pre_s = (
    component_times[[c for c in component_times.columns if c.endswith("_pre_step_s")]]
    .sum()
    .sum()
)
post_s = (
    component_times[[c for c in component_times.columns if c.endswith("_post_step_s")]]
    .sum()
    .sum()
)
write_cols = [c for c in task_times.columns if c.endswith(".Write_wall_time_s")]
solve_cols = [c for c in task_times.columns if c.endswith(".Solve_wall_time_s")]
write_s = task_times[write_cols].sum().sum()
solver_task_s = task_times[solve_cols].sum().sum()
solver_other_s = max(0.0, solver_task_s - refactorization_s - linear_solve_s)
stages = pd.Series(
    {
        "Component pre-step": pre_s,
        "Matrix refactorization (L/U combined)": refactorization_s,
        "Linear solve": linear_solve_s,
        "Other solver work": solver_other_s,
        "Component post-step": post_s,
        "Logger task (per-step)": write_s,
        "Logger final flush": float(performance.get("data_logger_stop_flush", 0.0)),
    }
).sort_values()

fig, ax = plt.subplots(figsize=(10, 5.5))
bars = ax.barh(stages.index, stages.values, color="#5b8db8")
ax.set_xlabel("Cumulative wall time (s)")
ax.set_title(
    f"Simulation time spent by stage — {performance['simulation_wall_clock']:.3f} s wall time"
)
ax.grid(axis="x", alpha=0.3)
for bar, value in zip(bars, stages.values):
    ax.text(
        value + max(stages.values) * 0.015,
        bar.get_y() + bar.get_height() / 2,
        f"{value:.4f} s",
        va="center",
    )
ax.set_xlim(0, max(stages.values) * 1.18)
fig.tight_layout()
plt.show()

# Compare the active simulation loop and end-to-end run for both logger modes.
logger_comparison = pd.DataFrame(
    {
        "CSV logger": {
            "Per-step logger task": csv_component_times.Data_logger_task_s.sum(),
            "Simulation calculation": csv_calculation_s,
            "Final file flush": float(csv_performance["data_logger_stop_flush"]),
            "End-to-end wall time": float(csv_performance["simulation_wall_clock"]),
        },
        "Real-time logger": {
            "Per-step logger task": component_times.Data_logger_task_s.sum(),
            "Simulation calculation": realtime_calculation_s,
            "Final file flush": float(performance["data_logger_stop_flush"]),
            "End-to-end wall time": float(performance["simulation_wall_clock"]),
        },
    }
)
logger_comparison
logger_comparison.plot(kind="bar", figsize=(11, 5), color=["#4c78a8", "#f58518"])
plt.ylabel("Wall time (s)")
plt.title("CSV vs. buffered real-time logger")
plt.xticks(rotation=15, ha="right")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

pd.DataFrame(
    {
        "cumulative_component_task_time_s": {
            "SSN GFM": gfm_component_s,
            "SSN PiLine": pi_line_component_s,
            "SSN base load": base_load_component_s,
            "SSN step load": step_load_component_s,
            "Both SSN loads": loads_component_s,
        }
    }
).sort_values("cumulative_component_task_time_s", ascending=False)

KLU performs lower- and upper-factor construction within a combined numerical
refactorization call, so separate L and U build timings are unavailable. The
component profile includes pre-step, post-step, and solver/integration time. The
real-time logger buffers scalar-expanded measurements in memory during the run
and writes its CSV at shutdown; the notebook compares logger-task, simulation,
flush, and end-to-end wall times.